In [3]:
import pandas as pd
from openai import OpenAI
import json
import time

# 1. Set up API
# ⚠️ Replace with your actual OpenRouter API Key
API_KEY = "REDACTED_API_KEY"
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY,
)

# 2. Read the data
df = pd.read_csv("ground_truth_25.csv")
print(f"Data loaded successfully. Total resumes: {len(df)}. Starting extraction...")

results = []

# 3. Loop through each resume
for index, row in df.iterrows():
    resume_text = row['resume_text']

    prompt = f"""
    You are a professional resume information extraction assistant.
    Please extract the following information from the resume text and output it strictly in JSON format.
    Required fields:
    - name (Fill with null if not found)
    - email (Fill with null if not found)
    - skills (List of skills, separated by semicolons. Fill with "" if not found)
    - years_experience (Years of experience. Numbers only. Fill with null if not found)

    Output ONLY the pure JSON code. Do not include any extra explanations or Markdown tags.

    Resume text:
    {resume_text}
    """

    max_retries = 3
    success = False

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                # ⚠️ 记得换成你新找的免费模型名字
                model="apodex/apodex-1.1-mini:free",
                messages=[
                    {"role": "system", "content": "You are a data extraction tool that only outputs JSON."},
                    {"role": "user", "content": prompt}
                ]
            )

            ai_text = response.choices[0].message.content.strip()
            ai_text = ai_text.replace("```json", "").replace("```", "").strip()
            parsed_data = json.loads(ai_text)

            result_row = row.to_dict()
            result_row['ai_name'] = parsed_data.get('name')
            result_row['ai_email'] = parsed_data.get('email')
            result_row['ai_skills'] = parsed_data.get('skills')
            result_row['ai_years'] = parsed_data.get('years_experience')
            results.append(result_row)

            print(f"Row {index + 1} extracted successfully: {parsed_data.get('name')}")
            success = True
            break # Break out of the retry loop upon success.

        except Exception as e:
            if "429" in str(e):
                print(f"Row {index + 1} rate limited. Waiting 10 seconds before retry... (Attempt {attempt+1})")
                time.sleep(10)
            else:
                print(f"Row {index + 1} failed: {e}")
                break # For any other error, do not retry; fail immediately.

    if not success:
        result_row = row.to_dict()
        result_row.update({"ai_name": "Error", "ai_email": "Error", "ai_skills": "Error", "ai_years": "Error"})
        results.append(result_row)

    # 4. Rest for a while after each successful extraction.
    time.sleep(5)

final_df = pd.DataFrame(results)
final_df.to_csv("llm_results.csv", index=False)
print("\nAll processing complete! Results saved as llm_results.csv")

Data loaded successfully. Total resumes: 25. Starting extraction...
Row 1 extracted successfully: Alice Wong
Row 2 extracted successfully: Bob
Row 3 extracted successfully: Sarah Lee
Row 4 extracted successfully: David
Row 5 extracted successfully: None
Row 6 extracted successfully: Tom
Row 7 extracted successfully: 王小明
Row 8 extracted successfully: Jessica
Row 9 extracted successfully: 李雷
Row 10 extracted successfully: Emily
Row 11 extracted successfully: 陈大文
Row 12 extracted successfully: John Doe
Row 13 extracted successfully: 赵敏
Row 14 extracted successfully: Mike
Row 15 extracted successfully: 王芳
Row 16 extracted successfully: John Smith
Row 17 extracted successfully: Emma Watson
Row 18 extracted successfully: None
Row 19 extracted successfully: Sarah
Row 20 extracted successfully: David
Row 21 extracted successfully: Chris Evans
Row 22 extracted successfully: Lisa
Row 23 extracted successfully: Alex Johnson
Row 24 extracted successfully: Rachel
Row 25 extracted successfully: dann

In [4]:
import pandas as pd

# 1. Read your standard answers and the results extracted by the AI.
true_df = pd.read_csv("ground_truth_25.csv")
ai_df = pd.read_csv("llm_results.csv")

# 2. Define a smart comparison function (handling cases involving letter casing, whitespace, and null values).
def is_match(true_val, ai_val):
    # Standardize to a string, trim leading and trailing whitespace, and convert to lowercase.
    t_str = str(true_val).strip().lower()
    a_str = str(ai_val).strip().lower()

    # If both are null, it counts as a match.
    if t_str in ["nan", "none", "null", ""] and a_str in ["nan", "none", "null", ""]:
        return True
    return t_str == a_str

# 3. Compare field by field and generate a match indicator column.
ai_df['name_match'] = ai_df.apply(lambda row: is_match(row['name_true'], row['ai_name']), axis=1)
ai_df['email_match'] = ai_df.apply(lambda row: is_match(row['email_true'], row['ai_email']), axis=1)
ai_df['skills_match'] = ai_df.apply(lambda row: is_match(row['skills_true'], row['ai_skills']), axis=1)
ai_df['years_match'] = ai_df.apply(lambda row: is_match(row['years_true'], row['ai_years']), axis=1)

# 4. Calculate the accuracy for each item.
total = len(ai_df)
name_acc = ai_df['name_match'].mean()
email_acc = ai_df['email_match'].mean()
skills_acc = ai_df['skills_match'].mean()
years_acc = ai_df['years_match'].mean()

# 5. Calculate the overall accuracy (counted as correct only if all four fields are correct).
overall_acc = (ai_df['name_match'] & ai_df['email_match'] & ai_df['skills_match'] & ai_df['years_match']).mean()

# 6. Print results
print("=" * 40)
print("EVALUATION RESULTS")
print("=" * 40)
print(f"Total records evaluated: {total}")
print("-" * 40)
print(f"Name Accuracy:   {name_acc:.2%}  ({ai_df['name_match'].sum()}/{total})")
print(f"Email Accuracy:  {email_acc:.2%}  ({ai_df['email_match'].sum()}/{total})")
print(f"Skills Accuracy: {skills_acc:.2%}  ({ai_df['skills_match'].sum()}/{total})")
print(f"Years Accuracy:  {years_acc:.2%}  ({ai_df['years_match'].sum()}/{total})")
print("-" * 40)
print(f"Overall Accuracy (All 4 fields correct): {overall_acc:.2%}")
print("=" * 40)

# 7. Save detailed comparison results.
ai_df.to_csv("evaluation_results.csv", index=False)
print("Detailed evaluation saved to evaluation_results.csv")

EVALUATION RESULTS
Total records evaluated: 25
----------------------------------------
Name Accuracy:   100.00%  (25/25)
Email Accuracy:  100.00%  (25/25)
Skills Accuracy: 64.00%  (16/25)
Years Accuracy:  88.00%  (22/25)
----------------------------------------
Overall Accuracy (All 4 fields correct): 52.00%
Detailed evaluation saved to evaluation_results.csv


In [5]:
import pandas as pd
import re

# 1. Read data
true_df = pd.read_csv("ground_truth_25.csv")
ai_df = pd.read_csv("llm_results.csv")

# 2. Define a function to clean a list of skills.
def clean_skills(s):
    if pd.isna(s) or str(s).strip().lower() in ['nan', 'none', 'null', '']:
        return set()
    # Remove square brackets, single quotes, and double quotes.去掉方括号、单引号、双引号
    s = str(s).replace('[', '').replace(']', '').replace("'", "").replace('"', "")
    # Split by semicolons, commas, or vertical bars.按分号、逗号或竖线分割
    items = re.split(r'[;,\|]', s)
    # Trim leading and trailing whitespace, convert to lowercase, filter out empty values, and remove duplicates.去除首尾空格，转小写，过滤空值，去重
    return set([item.strip().lower() for item in items if item.strip()])

# 3. Field-by-field comparison
def is_match(true_val, ai_val, is_skills=False):
    if is_skills:
        return clean_skills(true_val) == clean_skills(ai_val)

    t_str = str(true_val).strip().lower()
    a_str = str(ai_val).strip().lower()
    # Standardize null values
    if t_str in ["nan", "none", "null", ""] and a_str in ["nan", "none", "null", ""]:
        return True
    # Treat 0 and 0.0 as the same.
    if t_str.replace('.0','') == a_str.replace('.0',''):
        return True
    return t_str == a_str

ai_df['name_match'] = ai_df.apply(lambda row: is_match(row['name_true'], row['ai_name']), axis=1)
ai_df['email_match'] = ai_df.apply(lambda row: is_match(row['email_true'], row['ai_email']), axis=1)
ai_df['skills_match'] = ai_df.apply(lambda row: is_match(row['skills_true'], row['ai_skills'], is_skills=True), axis=1)
ai_df['years_match'] = ai_df.apply(lambda row: is_match(row['years_true'], row['ai_years']), axis=1)

# 4. Calculate the accuracy for each item.
total = len(ai_df)
name_acc = ai_df['name_match'].mean()
email_acc = ai_df['email_match'].mean()
skills_acc = ai_df['skills_match'].mean()
years_acc = ai_df['years_match'].mean()

overall_acc = (ai_df['name_match'] & ai_df['email_match'] & ai_df['skills_match'] & ai_df['years_match']).mean()

print("=" * 40)
print("EVALUATION RESULTS (Smart Matching)")
print("=" * 40)
print(f"Total records: {total}")
print("-" * 40)
print(f"Name Accuracy:   {name_acc:.2%}  ({ai_df['name_match'].sum()}/{total})")
print(f"Email Accuracy:  {email_acc:.2%}  ({ai_df['email_match'].sum()}/{total})")
print(f"Skills Accuracy: {skills_acc:.2%}  ({ai_df['skills_match'].sum()}/{total})")
print(f"Years Accuracy:  {years_acc:.2%}  ({ai_df['years_match'].sum()}/{total})")
print("-" * 40)
print(f"Overall Accuracy: {overall_acc:.2%}")
print("=" * 40)

ai_df.to_csv("evaluation_results_v2.csv", index=False)
print("Detailed results saved to evaluation_results_v2.csv")

EVALUATION RESULTS (Smart Matching)
Total records: 25
----------------------------------------
Name Accuracy:   100.00%  (25/25)
Email Accuracy:  100.00%  (25/25)
Skills Accuracy: 88.00%  (22/25)
Years Accuracy:  88.00%  (22/25)
----------------------------------------
Overall Accuracy: 76.00%
Detailed results saved to evaluation_results_v2.csv


In [6]:
import pandas as pd
import re

# 1. Read raw data and AI extraction results.
true_df = pd.read_csv("ground_truth_25.csv")
ai_df = pd.read_csv("llm_results.csv")

# ================= Guardrail insertion logic 插入护栏逻辑 =================
def apply_guardrails(row):
    # Guardrail 1: Validate email format 护栏1：校验邮箱格式
    if pd.notna(row['ai_email']):
        if not re.match(r"[^@]+@[^@]+\.[^@]+", str(row['ai_email'])):
            row['ai_email'] = "INVALID_EMAIL"

    # Guardrail 2: Soft-skill filtering (blacklist mechanism) 护栏2：过滤软技能（黑名单机制）
    if pd.notna(row['ai_skills']):
        soft_skills = ['communication', 'leadership', 'fast learner', 'teamwork', '沟通', '领导力']
        # Compatible with the list format `['Python', 'SQL']` or the string format `"Python; SQL"` that the AI ​​might return.
        skills_str = str(row['ai_skills']).replace('[', '').replace(']', '').replace("'", "").replace('"', "")
        skills = [s.strip().lower() for s in re.split(r'[;,]', skills_str) if s.strip()]

        # Remove soft skills from the blacklist.
        cleaned = [s for s in skills if s not in soft_skills]
        row['ai_skills'] = "; ".join(cleaned)

    return row

# 对 AI 结果应用护栏
ai_df = ai_df.apply(apply_guardrails, axis=1)
print("护栏应用完毕！已自动剔除软技能并校验邮箱格式。")
# ==============================================

# 2. Define the intelligent cleaning and matching function (your v2 version logic).
def clean_skills(s):
    if pd.isna(s) or str(s).strip().lower() in ['nan', 'none', 'null', '']:
        return set()
    s = str(s).replace('[', '').replace(']', '').replace("'", "").replace('"', "")
    items = re.split(r'[;,\|]', s)
    return set([item.strip().lower() for item in items if item.strip()])

def is_match(true_val, ai_val, is_skills=False):
    if is_skills:
        return clean_skills(true_val) == clean_skills(ai_val)
    t_str = str(true_val).strip().lower()
    a_str = str(ai_val).strip().lower()
    if t_str in ["nan", "none", "null", ""] and a_str in ["nan", "none", "null", ""]:
        return True
    if t_str.replace('.0','') == a_str.replace('.0',''):
        return True
    return t_str == a_str

# 3. Recalculate accuracy
ai_df['name_match'] = ai_df.apply(lambda row: is_match(row['name_true'], row['ai_name']), axis=1)
ai_df['email_match'] = ai_df.apply(lambda row: is_match(row['email_true'], row['ai_email']), axis=1)
ai_df['skills_match'] = ai_df.apply(lambda row: is_match(row['skills_true'], row['ai_skills'], is_skills=True), axis=1)
ai_df['years_match'] = ai_df.apply(lambda row: is_match(row['years_true'], row['ai_years']), axis=1)

# 4. Print results
total = len(ai_df)
print("=" * 40)
print("EVALUATION RESULTS (WITH GUARDRAILS)")
print("=" * 40)
print(f"Name Accuracy:   {ai_df['name_match'].mean():.2%}")
print(f"Email Accuracy:  {ai_df['email_match'].mean():.2%}")
print(f"Skills Accuracy: {ai_df['skills_match'].mean():.2%}")
print(f"Years Accuracy:  {ai_df['years_match'].mean():.2%}")
print("-" * 40)
print(f"Overall Accuracy: {(ai_df['name_match'] & ai_df['email_match'] & ai_df['skills_match'] & ai_df['years_match']).mean():.2%}")
print("=" * 40)

# 5. Save the final results (for report writing)
ai_df.to_csv("evaluation_results_v3.csv", index=False)
print("Final results saved to evaluation_results_v3.csv")

护栏应用完毕！已自动剔除软技能并校验邮箱格式。
EVALUATION RESULTS (WITH GUARDRAILS)
Name Accuracy:   100.00%
Email Accuracy:  100.00%
Skills Accuracy: 100.00%
Years Accuracy:  88.00%
----------------------------------------
Overall Accuracy: 88.00%
Final results saved to evaluation_results_v3.csv
